# Six-agent long training: 9,000 m overtaking task

Based on `sensor_dqn_long_training.ipynb`, extended with the
adaptive-reservoir and prototype experiments. This new notebook and its supporting
code live in **final_project/**. Existing notebooks, trained models, and legacy
environment defaults remain unchanged.

**Protocol: 500 training episodes × six agents × three training seeds = 9,000
training episodes**, followed by 1,080 held-out trips. No prior task policies are
transferred into this study. The two prototype variants additionally share a
20-episode training-only observation corpus per seed, reused over three passes.

Select **MORL4VSO (.venv)**. The full run is managed by background worker processes;
closing this notebook does not stop training. Run the progress cells again to
refresh them. Status is explicit: partial results do not mean the full study is complete.

Run All reviews the available historical artifacts. Enable validation and the pilot in the setup cell before launching a new study.

## Runs stopped

Training was stopped at the user's request. Saved checkpoints, evaluation results, and replays are preserved. Automatic launch is disabled in this notebook; refreshing progress/results will not restart training. STOP markers remain in the study directories.


## Preliminary budget update

**500 episodes per agent per training seed: 6 × 3 × 500 = 9,000 training episodes total.**
The earlier 5,000-episode study was stopped and preserved separately. All preliminary models start fresh, with epsilon decaying from 1.0 to 0.1 over 500 episodes. This prevents mixing the old, mostly exploratory checkpoints with a shorter training schedule.

Traffic, reward, gamma, neural settings, three training seeds, and held-out evaluations remain unchanged. The existing feasibility validation and completed pipeline pilot are reused after source/configuration checks. Results go to `artifacts/overtaking_9000m_500ep_v1/`.


In [1]:
from pathlib import Path
import os, sys, json
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "experiments/run_9000m_study.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from experiments.run_9000m_study import (POLICIES, TRAIN_SEEDS,
    prepare, read_protocol, validate_scenario, coordinate, launch, summarize, plot_results)
from experiments.run_9000m_preliminary import DEFAULT_OUTPUT, prepare_preliminary
OUTPUT = DEFAULT_OUTPUT
RECORDED = ROOT / "runs/exploratory/overtaking_9000m_500ep_v1"
RUN_VALIDATION_AND_PILOT = False
LAUNCH_OR_RESUME = False
prepare_preliminary(OUTPUT)
protocol = read_protocol(OUTPUT)
print("Project:", ROOT)
print("Study:", OUTPUT)
print("Interpreter:", sys.executable)
display(pd.Series({"route_m": protocol["config"]["reward"]["route_distance_m"],
                   "deadline_s": protocol["config"]["highway"]["duration"],
                   "episodes_per_agent_per_seed": protocol["episodes"],
                   "training_seeds": protocol["training_seeds"],
                   "trained_models": len(POLICIES)*len(TRAIN_SEEDS),
                   "total_training_episodes": protocol["episodes"]*len(POLICIES)*len(TRAIN_SEEDS),
                   "held_out_trips_per_model": len(protocol["test_seeds"])*len(protocol["weather"]),
                   "gamma": protocol["gamma"]}))

Project: /Users/zac/Desktop/Reinforcment_Learning/final_project
Study: /Users/zac/Desktop/Reinforcment_Learning/final_project/experiments/artifacts/overtaking_9000m_500ep_v1
Interpreter: /Users/zac/Desktop/Reinforcment_Learning/final_project/.venv/bin/python


route_m                                    9000.0
deadline_s                                    360
episodes_per_agent_per_seed                   500
training_seeds                 [42, 10042, 20042]
trained_models                                 18
total_training_episodes                      9000
held_out_trips_per_model                       60
gamma                                       0.999
dtype: object

## 1. Traffic and task changes

The route is **9,000 m with a 360 s deadline**. Average speed must exceed 25 m/s
once maneuvering and acceleration are included. Cruising at 20 m/s takes 450 s.
Ego target speeds are 10, 15, 20, 25, and 30 m/s, preserving the five driving action
names while permitting safe slowing behind traffic. The sensing menu is unchanged.

Three lanes contain staggered, slower 18–22 m/s IDM vehicles. Traffic count is determined by the moving window and spacing, rather than the legacy `vehicles_count` setting. Background vehicles
hold their lanes and can brake for traffic. Initial same-lane spacing is about
500 m with lane offsets and random position/speed variation. Vehicles are maintained
in a moving window: newly activated traffic enters 300–600 m ahead, outside the
150 m sensor radius, with a minimum 55 m same-lane placement separation. Distant
vehicles more than 100 m behind are removed. Initial placements may be closer,
including a leader roughly 90 m ahead in the ego lane.

Each seed draws immutable traffic identities and nominal trajectories once.
Off-window traffic follows its nominal trajectory; active traffic uses IDM.
This is a controlled traffic approximation, not a full city-scale simulation.
Encountered cars can be overtaken repeatedly along the route. Counts credit each
vehicle once after it was ahead within 150 m and the ego clears it by 10 m.
Counters and truth-based controller diagnostics are excluded from policy features.

Sensors, weather, 100 Hz physics, 2 Hz decisions, and observation features remain
shared across methods. Learned policies never receive true gaps, lane IDs, vehicle
IDs, or the oracle's commands.

## 2. Agents and learning settings

| Agent | Learning |
|---|---|
| Expected SARSA | Linear Q features, normalized semi-gradient updates |
| DQN | ReLU Q network, replay and frozen target network |
| Dyna-DQN | DQN plus learned one-step transition/reward/termination model and planning |
| Adaptive reservoir | Fixed signed hidden synapses, threshold adaptation within each observation, learned Q readout |
| Prototype control | Fixed positive normalized weights, inhibitory feedback and threshold calibration, learned Q readout |
| STDP-pretrained prototypes | Matched prototype setup with unsupervised STDP, then frozen representation and learned Q readout |

All agents receive 500 training episodes, gamma **0.999**, and epsilon **1.0 → 0.1**.
Training seed blocks are **42–541**, **10042–10541**, and **20042–20541**, avoiding
training overlap. Readout/network optimizer choices retain the earlier implementation;
linear SARSA uses alpha 0.12, neural methods use Adam at 0.001. SNNs retain 64 hidden
neurons and a 16-step internal window. Prototype training sees the same separate
random-policy corpus/order for each seed; it is collected once and reused. These
methods therefore have extra pretraining observations, reported separately.

Reward coefficients are shared: progress 0.02/m, time penalty 0.1/s, sensor energy
penalty 0.001/J, uncertainty penalty 0.1/s, completion bonus 100, early finish bonus
up to 25, and failure penalty 200. The original highway speed/collision reward is
zero to avoid double-counting. A crash or missed deadline has no bootstrap term.
Gamma 0.999 retains about 49% of a reward 720 decisions ahead; 0.99 would retain
less than 0.1%. The larger terminal bonuses/penalties address long-route credit
assignment and make early termination unattractive.

The prototypes are Diehl–Cook-inspired current-based LIF models, not a full
replication. Their calibrated weights/thresholds freeze before Q learning.
The adaptive reservoir resets threshold offsets per observation and adapts locally.
There is no hidden-state mutation during next-state target evaluation.

## 3. Feasibility gate

Before full training, test five **development-only** traffic seeds. A privileged
scripted overtaking controller must complete at least four without any crashes;
completed routes must include at least three overtakes. A lane-following controller
and repeated SLOWER baseline must fail to complete. The oracle only establishes
feasibility; it is not a sensor-based competitor, teacher, or source of demonstrations.
A failed gate prevents the full run from starting. The source/config hashes bind
the gate to this exact scenario.

In [2]:
validation_file = OUTPUT / "validation.json"
if RUN_VALIDATION_AND_PILOT:
    if not validation_file.exists() or not json.loads(validation_file.read_text())["passed"]:
        validate_scenario(OUTPUT)
validation_directory = OUTPUT if validation_file.is_file() else RECORDED
if (validation_directory / "scenario_validation.csv").is_file():
    display(pd.read_csv(validation_directory / "scenario_validation.csv"))
    display(json.loads((validation_directory / "validation.json").read_text()))


,controller,seed,return,crashed,completed,deadline_failure,completion_time_s,elapsed_s,distance_m,steps,...,overtakes,lane_changes,following_s,traffic_spawned,mode_fraction_all_on,drive_fraction_LANE_RIGHT,drive_fraction_FASTER,drive_fraction_IDLE,drive_fraction_LANE_LEFT,drive_fraction_SLOWER
0,oracle,900001,177.413196,False,True,False,326.0,326.0,9009.852002,652,...,14,8,64.62,17,1.0,0.006135,0.119632,0.753067,0.006135,0.115031
1,follow,900001,-177.782177,False,False,True,NaN,360.0,6887.700530,720,...,0,0,291.30,4,1.0,NaN,0.494444,0.197222,NaN,0.308333
2,slow,900001,-243.895348,False,False,True,NaN,360.0,3619.000000,720,...,0,0,0.00,4,1.0,NaN,NaN,NaN,NaN,1.000000
3,oracle,900002,-143.221388,False,False,True,NaN,360.0,8622.822550,720,...,9,7,215.61,11,1.0,0.005556,0.138889,0.720833,0.004167,0.130556
4,follow,900002,-171.772526,False,False,True,NaN,360.0,7171.688578,720,...,0,0,318.54,6,1.0,NaN,0.381944,0.343056,NaN,0.275000
5,slow,900002,-243.894623,False,False,True,NaN,360.0,3619.000000,720,...,0,0,0.00,4,1.0,NaN,NaN,NaN,NaN,1.000000
6,oracle,900003,181.343324,False,True,False,316.0,316.0,9003.912074,632,...,18,13,33.76,23,1.0,0.009494,0.069620,0.857595,0.011076,0.052215
7,follow,900003,-175.264517,False,False,True,NaN,360.0,7026.688460,720,...,0,0,318.49,5,1.0,NaN,0.495833,0.098611,NaN,0.405556
8,slow,900003,-243.895386,False,False,True,NaN,360.0,3619.000000,720,...,0,0,0.00,4,1.0,NaN,NaN,NaN,NaN,1.000000
9,oracle,900004,187.318462,False,True,False,300.5,300.5,9004.086799,601,...,19,14,0.00,21,1.0,0.011647,0.003328,0.971714,0.011647,0.001664


{'passed': True,
 'oracle_completion': 0.8,
 'oracle_crashes': 0,
 'updated_at': '2026-11-18T12:43:41.859431-05:00',
 'config_sha256': 'e288e878dfd28ce80e40b4a5670fb30f15bd6190793de5a4c8311ea1a99f3f8a',
 'backend_sha256': '35ec0b0a63dbe890b8f4649e8bc0cf80a1b80099530d004ee33a99129fa3529f'}

## 4. Short pipeline pilot

Run all six algorithms for **three episodes each** on a separate development seed,
then test them on two development-only scenarios. This checks integration and saved
policy replay, not convergence. Pilot models are not used to initialize the full run.
Full training is gated on a successful pilot execution, not on pilot performance.

In [3]:
pilot_output = OUTPUT / "pilot"
pilot_status = pilot_output / "study_status.json"
if RUN_VALIDATION_AND_PILOT:
    if not pilot_status.exists() or json.loads(pilot_status.read_text())["phase"] != "complete":
        prepare(pilot_output, episodes=3, seeds=[70042], pilot=True)
        coordinate(pilot_output, workers=3)
pilot_results = pilot_output if pilot_status.is_file() else RECORDED / "pilot"
if (pilot_results / "summary.csv").is_file():
    display(pd.read_csv(pilot_results / "summary.csv"))


{'phase': 'complete',
 'failures': [],
 'updated_at': '2026-11-18T12:51:55.113556-05:00'}

,policy,training_seeds,completion,completion_sd,crash,deadline_failure,overtakes,power_w,following_s,mean_return,successful_time_s
0,Adaptive reservoir,1,0.0,NaN,1.0,0.0,1.0,123.000000,0.0,-192.906506,NaN
1,DQN,1,0.0,NaN,1.0,0.0,1.0,16.996104,0.0,-192.903848,NaN
2,Dyna-DQN,1,0.0,NaN,1.0,0.0,7.5,37.945419,0.0,-155.799440,NaN
3,Expected SARSA,1,0.0,NaN,0.0,1.0,0.0,6.687500,0.0,-184.387036,NaN
4,Prototype control (no STDP),1,0.0,NaN,1.0,0.0,0.0,123.000000,0.0,-197.246541,NaN
5,STDP-pretrained prototypes,1,0.0,NaN,1.0,0.0,0.0,123.000000,0.0,-197.716376,NaN


## 5. Launch or resume 500-episode training

Three single-threaded CPU workers process 18 policy/seed jobs. Training can take
many hours or days depending on episode survival and hardware. The runner writes
`status.json` every episode and checkpoints every 25 episodes. Completed models
are immediately evaluated and replayed while the other jobs continue.

Each local `resume.pkl` is a **trusted Python checkpoint** containing the agent,
Adam state, replay buffer, Dyna model, completed episode history, and random states.
Only load checkpoints produced by this project. Portable `agent.pt`/`agent.npz`
files are for inference. Source/config changes are rejected rather than mixed into
an existing study. Job locks prevent simultaneous writers.

The coordinator runs independently of the notebook. To resume after interruption:

```bash
.venv/bin/python -m experiments.run_9000m_preliminary --launch
```

For a terminal status report:

```bash
.venv/bin/python -m experiments.run_9000m_preliminary --status
```

To stop a running study gracefully at episode/checkpoint boundaries, create a file
named `STOP` inside the study directory. Remove it before resuming. Training data
already checkpointed remain available.

In [4]:
status_path = OUTPUT / "study_status.json"
state = json.loads(status_path.read_text()) if status_path.exists() else {}
alive = False
if state.get("phase") == "running":
    try:
        os.kill(state["pid"], 0)
        alive = True
    except ProcessLookupError:
        pass
if LAUNCH_OR_RESUME and state.get("phase") != "complete" and not alive:
    if not RUN_VALIDATION_AND_PILOT:
        raise ValueError("Enable RUN_VALIDATION_AND_PILOT to validate the local protocol before launch.")
    launch(OUTPUT, workers=3)
else:
    print("Study state:", state.get("phase", "not launched"), "| running:", alive)
print("Log:", OUTPUT / "runner.log")
print("Refresh the cells below to view progress; this notebook does not wait for all training.")

Study state: running | running: True
Log: /Users/zac/Desktop/Reinforcment_Learning/final_project/experiments/artifacts/overtaking_9000m_500ep_v1/runner.log
Refresh the cells below to view progress; this notebook does not wait for all training.


## 6. Progress and partial results

Each row is one policy and training seed. Evaluation is 20 held-out traffic seeds
across clear, rain, and fog: **60 trips per model**, **1,080 trips total**. Missing
models remain pending. Completion time is conditional on success. Report crashes,
deadline failures, passing, following, and sensor energy together; early crashes
can reduce energy and exposure. Shared weather/traffic trials are not independent
training replicates. Equal episode budgets do not imply equal steps or compute.

In [5]:
RESULTS = OUTPUT if any(OUTPUT.glob("seed*")) else RECORDED
rows = []
for seed in protocol["training_seeds"]:
    for policy, label in POLICIES.items():
        folder = RESULTS / f"seed{seed}" / policy
        status = folder / "status.json"
        current = json.loads(status.read_text()) if status.exists() else {"phase": "queued", "episode": 0}
        rows.append({"policy": label, "training_seed": seed,
                     "phase": current["phase"], "episode": current.get("episode", 0),
                     "target": protocol["episodes"], "updated_at": current.get("updated_at", "")})
progress = pd.DataFrame(rows)
display(progress)
print("Recorded training episodes:", progress.episode.sum(), "/", 9000)
if RESULTS == OUTPUT:
    summary = summarize(OUTPUT)
else:
    summary = pd.read_csv(RESULTS / "summary.csv") if (RESULTS / "summary.csv").exists() else None
if summary is None:
    print("No full-model evaluation has completed yet.")
else:
    display(summary.round(3))
    if (RESULTS / "per_training_seed.csv").is_file():
        display(pd.read_csv(RESULTS / "per_training_seed.csv").round(3))
if (OUTPUT / "study_status.json").exists():
    display(json.loads((OUTPUT / "study_status.json").read_text()))

,policy,training_seed,phase,episode,target,updated_at
0,Expected SARSA,42,training,4,500,2026-11-18T14:13:19.773100-05:00
1,DQN,42,training,6,500,2026-11-18T14:13:24.116949-05:00
2,Dyna-DQN,42,training,4,500,2026-11-18T14:13:23.146205-05:00
3,Adaptive reservoir,42,queued,0,500,
4,Prototype control (no STDP),42,queued,0,500,
5,STDP-pretrained prototypes,42,queued,0,500,
6,Expected SARSA,10042,queued,0,500,
7,DQN,10042,queued,0,500,
8,Dyna-DQN,10042,queued,0,500,
9,Adaptive reservoir,10042,queued,0,500,


Recorded training episodes: 14 / 9000
No full-model evaluation has completed yet.


{'phase': 'running',
 'pid': 30531,
 'workers': 3,
 'started_at': '2026-11-18T14:12:38.761361-05:00'}

In [6]:
if (OUTPUT / "summary.csv").exists():
    plot_results(OUTPUT)
    display(Image(filename=str(OUTPUT / "comparison.png")))
else:
    print("Comparison figure will appear after the first full-model evaluation.")
if (OUTPUT / "training.csv").exists():
    import matplotlib.pyplot as plt
    history = pd.read_csv(OUTPUT / "training.csv")
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    for label, group in history.groupby("policy"):
        means = group.groupby("episode")["return"].mean().rolling(100, min_periods=1).mean()
        completed = group.groupby("episode")["completed"].mean().rolling(100, min_periods=1).mean()
        axes[0].plot(means.index, means, label=label)
        axes[1].plot(completed.index, completed, label=label)
    axes[0].set(title="Training return: 100-episode mean", xlabel="Episode", ylabel="Return")
    axes[1].set(title="Training completion: 100-episode mean", xlabel="Episode", ylabel="Completion fraction", ylim=(-.05, 1.05))
    axes[0].legend(fontsize=8)
    fig.savefig(OUTPUT / "training_curves.png", dpi=160)
    display(fig)
    plt.close(fig)

Comparison figure will appear after the first full-model evaluation.


## 7. Replays and saved artifacts

Every finished model automatically gets an interactive replay and GIF for the
same first held-out scenario (clear, traffic seed 100000). If its first failure
is another scenario, save that too. Replays are regenerated with frozen policies
and checked against the stored evaluation metrics. They show simulator road truth
separately from the sensor evidence available to the policy. Frames are captured
at decision boundaries, every 0.5 seconds.

Each model directory contains a training CSV, portable policy, resumable state,
status, evaluation-progress CSV, final evaluation CSV, and replays. Aggregate CSVs,
plots, protocol/configuration, validation/pilot records, and a report live in the
study root. Prototype corpora are stored once per training seed.

In [7]:
replays = sorted(OUTPUT.glob("seed*/*/replays/*/replay.html"))
if replays:
    for replay in replays:
        relative = os.path.relpath(replay, ROOT / "experiments")
        display(Markdown(f"[{replay.parent.parent.parent.parent.name} / {replay.parent.parent.parent.name} / {replay.parent.name}]({relative})"))
else:
    print("Full-study replays are pending. Pilot replays can be inspected under:", OUTPUT / "pilot")
report = OUTPUT / "study_report.md"
if report.exists():
    display(Markdown(report.read_text()))
else:
    print("The final report will be created when all 18 models have completed.")

Full-study replays are pending. Pilot replays can be inspected under: /Users/zac/Desktop/Reinforcment_Learning/final_project/experiments/artifacts/overtaking_9000m_500ep_v1/pilot
The final report will be created when all 18 models have completed.


## DQN scenario-inspection replays

- [Longest-duration training GIF (360 s)](artifacts/overtaking_9000m_500ep_v1/scenario_replays/dqn_longest_training/policy.gif) · [interactive viewer](artifacts/overtaking_9000m_500ep_v1/scenario_replays/dqn_longest_training/replay.html) · [six-frame overview](artifacts/overtaking_9000m_500ep_v1/scenario_replays/dqn_longest_training/scenario_overview.png).
- [Longest completed-model evaluation GIF (10.5 s)](artifacts/overtaking_9000m_500ep_v1/scenario_replays/dqn_longest_evaluation/policy.gif) · [interactive viewer](artifacts/overtaking_9000m_500ep_v1/scenario_replays/dqn_longest_evaluation/replay.html).

The training replay reconstructs episode 2 (traffic seed 43) from initialization, including all learning updates, and matches its saved metrics. It used epsilon 0.9982 (almost entirely random actions), covered 6,296 m, and missed the deadline without crashing. It is not a frozen learned-policy performance example. There were 3–4 active background vehicles in this trajectory; traffic pulls ahead as the ego slows. The road viewport is widened for inspection without changing the scenario or sensor observations.

The evaluation replay uses the frozen 500-episode DQN (training seed 42), clear weather and traffic seed 100007. It crashes after 10.5 s / 312 m. Both exports were checked against saved records, and the original policy checkpoint stayed unchanged.
